In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuration for high-quality scientific plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.family'] = 'sans-serif'
plt.switch_backend('Agg') # Background processing (no GUI needed)

# Path Configuration
DATA_PATH = "Project_Data"
RESULTS_FILE = "Audio_Features_Extracted.csv"

def extract_comprehensive_features(file_path):
    """
    Extracts a comprehensive set of audio features:
    - Time-domain: ZCR, RMS Energy
    - Frequency-domain: Spectral Centroid, Rolloff, Flux
    - Cepstral: MFCCs (13 coefficients)
    - Prosodic: Fundamental Frequency (F0/Pitch)
    """
    try:
        # Load audio file
        y, sr = librosa.load(file_path, sr=None)
        
        # 1. Root Mean Square Energy (Volume/Intensity)
        rms = np.mean(librosa.feature.rms(y=y))
        
        # 2. Zero Crossing Rate (Noise/Fricatives indicator)
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=y))
        
        # 3. Spectral Centroid (Brightness of sound)
        spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        
        # 4. Spectral Rolloff (High frequency energy)
        spec_roll = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        
        # 5. MFCCs (Timbre - first 13 coefficients)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)
        
        # 6. Fundamental Frequency (Pitch) using PYIN algorithm
        f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), 
                                fmax=librosa.note_to_hz('C7'), 
                                frame_length=2048)
        pitch_mean = np.nanmean(f0) if np.any(~np.isnan(f0)) else 0
        
        # Aggregate features
        features = [rms, zcr, spec_cent, spec_roll, pitch_mean] + list(mfccs_mean)
        return features
        
    except Exception as e:
        # Log error silently
        return None

def main():
    # Define Column Names
    mfcc_cols = [f'MFCC_{i+1}' for i in range(13)]
    columns = ['Emotion', 'RMS_Energy', 'ZCR', 'Spectral_Centroid', 
               'Spectral_Rolloff', 'Pitch_Mean'] + mfcc_cols
    
    data_list = []
    
    # --- 1. DATA INGESTION & FEATURE EXTRACTION ---
    if not os.path.exists(DATA_PATH):
        print(f"[ERROR] Directory {DATA_PATH} not found.")
        return

    print("[INFO] Starting Feature Extraction Pipeline...")
    
    for emotion_folder in os.listdir(DATA_PATH):
        folder_path = os.path.join(DATA_PATH, emotion_folder)
        
        if not os.path.isdir(folder_path): continue
            
        # Label Encoding based on folder name
        if "angry" in emotion_folder.lower(): label = "Angry"
        elif "sad" in emotion_folder.lower(): label = "Sad"
        else: continue 
        
        wav_files = glob.glob(os.path.join(folder_path, "*.wav"))
        print(f"[INFO] Processing Class: {label} | Found {len(wav_files)} samples.")
        
        for wav in wav_files:
            feats = extract_comprehensive_features(wav)
            if feats:
                data_list.append([label] + feats)

    # Convert to DataFrame
    df = pd.DataFrame(data_list, columns=columns)
    df.to_csv(RESULTS_FILE, index=False)
    print(f"[SUCCESS] Data extraction complete. Saved to {RESULTS_FILE}")

    # --- 2. STATISTICAL ANALYSIS & VISUALIZATION ---
    if df.empty: return

    print("[INFO] Generating Statistical Visualizations...")

    # A. Feature Distributions (KDE Plots)
    # Visualizing the density of Energy and Pitch
    fig1, ax = plt.subplots(1, 2, figsize=(16, 6))
    sns.kdeplot(data=df, x='RMS_Energy', hue='Emotion', fill=True, palette='Set1', ax=ax[0])
    ax[0].set_title('Kernel Density Estimation: Signal Energy')
    
    sns.kdeplot(data=df, x='Pitch_Mean', hue='Emotion', fill=True, palette='Set1', ax=ax[1])
    ax[1].set_title('Kernel Density Estimation: Fundamental Frequency (F0)')
    plt.savefig('Fig1_Feature_Distributions.png', dpi=300)
    
    # B. Violin Plots (Advanced Boxplots)
    # Comparing Spectral Centroid (Timbre Brightness)
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='Emotion', y='Spectral_Centroid', data=df, palette='Set1')
    plt.title('Spectral Centroid Distribution (Violin Plot)')
    plt.savefig('Fig2_Spectral_Analysis.png', dpi=300)

    # C. Correlation Matrix (Heatmap)
    # Analyzing how features correlate with each other
    plt.figure(figsize=(12, 10))
    # Select numeric columns only excluding Emotion label
    numeric_df = df.drop(columns=['Emotion'])
    corr_matrix = numeric_df.corr()
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
    plt.title('Feature Correlation Matrix')
    plt.savefig('Fig3_Correlation_Heatmap.png', dpi=300)

    # D. PCA Analysis (Dimensionality Reduction)
    # Projecting multi-dimensional data into 2D space
    print("[INFO] Performing Principal Component Analysis (PCA)...")
    x = df.drop(columns=['Emotion']).values
    x = StandardScaler().fit_transform(x) # Normalize features
    
    pca = PCA(n_components=2)
    principalComponents = pca.fit_transform(x)
    pca_df = pd.DataFrame(data=principalComponents, columns=['PC1', 'PC2'])
    pca_df['Emotion'] = df['Emotion']
    
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x='PC1', y='PC2', hue='Emotion', data=pca_df, palette='Set1', s=100, alpha=0.7)
    plt.title(f'PCA Projection (Explained Variance: {pca.explained_variance_ratio_.sum():.2f})')
    plt.savefig('Fig4_PCA_Cluster_Analysis.png', dpi=300)
    
    print("[SUCCESS] All visualizations generated and saved.")

if __name__ == "__main__":
    main()

[INFO] Starting Feature Extraction Pipeline...
[INFO] Processing Class: Angry | Found 200 samples.
[INFO] Processing Class: Sad | Found 200 samples.
[SUCCESS] Data extraction complete. Saved to Audio_Features_Extracted.csv
[INFO] Generating Statistical Visualizations...


C:\Users\jimgt\AppData\Local\Temp\ipykernel_1856\3929911619.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='Emotion', y='Spectral_Centroid', data=df, palette='Set1')


[INFO] Performing Principal Component Analysis (PCA)...
[SUCCESS] All visualizations generated and saved.
